In [ ]:
# ✅ WEDAFALL Fall Detection with YOLOv8 Classifier - FIXED

# ✅ Step 1: Install Required Libraries (already done via command line)
# For reference: pip install --no-cache-dir numpy==1.23.5 pandas==2.0.3 torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cpu ultralytics==8.0.197 scikit-learn matplotlib opencv-python pyyaml

# ✅ Step 2: Import Libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO
from IPython.display import display, Markdown
import logging
import requests
import torch
import yaml

# ✅ Step 3: Log Versions for Debugging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.info(f"Python version: {sys.version}")
logging.info(f"NumPy version: {np.__version__}")
logging.info(f"Pandas version: {pd.__version__}")
logging.info(f"Torch version: {torch.__version__}")
logging.info(f"CUDA available: {torch.cuda.is_available()}")

# ✅ Step 4: Configuration
DATASET_PATH = r"dataset"  # Updated to absolute path
CLASS_NAMES = ['fall', 'normal']
MODEL_CONFIG = "yolov8n-cls.pt"  # Pre-trained classification model
PROJECT_NAME = "wedafall_yolo"
EXPERIMENT_NAME = "fall_classifier"
IMG_SIZE = 224
EPOCHS = 30
BATCH = 16

# ✅ Step 5: Download Model if Missing
def download_yolo_model(model_name="yolov8n-cls.pt"):
    """Download YOLOv8 model if not present."""
    model_path = Path(model_name)
    if not model_path.exists():
        logging.info(f"Downloading {model_name}...")
        url = f"https://github.com/ultralytics/assets/releases/download/v0.0.0/{model_name}"
        try:
            response = requests.get(url)
            response.raise_for_status()
            with open(model_path, 'wb') as f:
                f.write(response.content)
            logging.info(f"Downloaded {model_name} successfully.")
        except Exception as e:
            logging.error(f"Failed to download {model_name}: {e}")
            raise
    return model_path

# ✅ Step 6: Verify Dataset Structure
def verify_dataset(dataset_path):
    """Verify dataset structure and data.yaml existence."""
    dataset_path = Path(dataset_path)
    data_yaml = dataset_path / "data.yaml"
    required_dirs = [
        dataset_path / "train" / "fall",
        dataset_path / "train" / "normal",
        dataset_path / "val" / "fall",
        dataset_path / "val" / "normal",
        dataset_path / "test" / "fall",
        dataset_path / "test" / "normal"
    ]
    
    if not data_yaml.exists():
        logging.warning(f"data.yaml not found in {dataset_path}. Creating one...")
        create_data_yaml(dataset_path)
    
    for dir_path in required_dirs:
        if not dir_path.exists():
            logging.error(f"Directory {dir_path} does not exist.")
            raise FileNotFoundError(f"Directory {dir_path} does not exist.")
        logging.info(f"Verified directory: {dir_path}")
    
    logging.info("Dataset structure verified successfully.")
    return data_yaml

# FIXED: Create data.yaml with proper format for YOLOv8 classification
def create_data_yaml(dataset_path):
    """Create a basic data.yaml file for YOLOv8 classification."""
    dataset_path = Path(dataset_path).resolve()  # Get absolute path
    data_yaml = dataset_path / "data.yaml"
    
    # For YOLOv8 classification, data.yaml should contain the dataset root path
    # YOLOv8 will automatically look for train/, val/, test/ subdirectories
    data_yaml_content = {
        "path": str(dataset_path),  # Root dataset directory
        "train": "train",  # Relative to path
        "val": "val",      # Relative to path
        "test": "test",    # Relative to path
        "nc": len(CLASS_NAMES),  # Number of classes
        "names": CLASS_NAMES     # Class names
    }
    
    with open(data_yaml, 'w') as f:
        yaml.dump(data_yaml_content, f, default_flow_style=False)
    
    logging.info(f"Created data.yaml at {data_yaml}")
    logging.info(f"Dataset path: {dataset_path}")
    
    return data_yaml

# ✅ Step 7: Initialize and Train Model
try:
    data_yaml = verify_dataset(DATASET_PATH)
except FileNotFoundError as e:
    logging.error(f"Dataset verification failed: {e}")
    raise

# Download model if not present
try:
    download_yolo_model(MODEL_CONFIG)
except Exception as e:
    logging.error(f"Model download failed: {e}")
    raise

logging.info("Initializing YOLOv8 model...")
try:
    model = YOLO(MODEL_CONFIG)
except Exception as e:
    logging.error(f"Failed to initialize YOLO model: {e}")
    raise

logging.info("Starting model training...")
try:
    results = model.train(
        data=str(DATASET_PATH),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        project=PROJECT_NAME,
        name=EXPERIMENT_NAME,
        device='cpu',  # Explicitly use CPU
        patience=10,  # Early stopping
        verbose=True
        cache=False 
    )
except Exception as e:
    logging.error(f"Training failed: {e}")
    raise

# ✅ Step 8: Load Trained Model for Inference
best_model_path = f"{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt"
if not Path(best_model_path).exists():
    logging.error(f"Best model not found at {best_model_path}.")
    raise FileNotFoundError(f"Best model not found at {best_model_path}")

logging.info(f"Loading best model from {best_model_path}...")
model = YOLO(best_model_path)

# ✅ Step 9: Evaluate on Test Set
test_dir = Path(DATASET_PATH) / "test"
all_images = list(test_dir.rglob("*.jpg")) + list(test_dir.rglob("*.png"))  # Include PNG files too

if not all_images:
    logging.error("No images found in test directory.")
    raise ValueError(f"No .jpg or .png images found in {test_dir}")

y_true, y_pred = [], []
for img_path in all_images:
    try:
        result = model.predict(source=str(img_path), save=False, verbose=False, device='cpu')
        pred_class = result[0].probs.top1
        y_pred.append(pred_class)
        
        # Determine true class based on directory structure
        if 'fall' in str(img_path.parent).lower():
            y_true.append(0)  # fall class
        else:
            y_true.append(1)  # normal class
            
    except Exception as e:
        logging.warning(f"Error processing image {img_path}: {e}")
        continue

# ✅ Step 10: Metrics and Report
if not y_true or not y_pred:
    logging.error("No predictions made. Check test dataset or model inference.")
    raise ValueError("No predictions made.")

logging.info("Generating classification report...")
print("\n📊 Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

accuracy = np.mean(np.array(y_true) == np.array(y_pred))
print(f"✅ Accuracy: {accuracy * 100:.2f}%")
print(f"✅ Best model saved at: {best_model_path}")

# ✅ Step 11: Display Training Results
print("\n📈 Training Results Summary:")
print(f"Project: {PROJECT_NAME}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Epochs: {EPOCHS}")
print(f"Image Size: {IMG_SIZE}")
print(f"Batch Size: {BATCH}")
print(f"Device: CPU")
print(f"Classes: {CLASS_NAMES}")
print(f"Total Test Images: {len(all_images)}")

# Optional: Show sample predictions
print("\n🔍 Sample Predictions:")
sample_size = min(5, len(all_images))
for i, img_path in enumerate(all_images[:sample_size]):
    try:
        result = model.predict(source=str(img_path), save=False, verbose=False, device='cpu')
        pred_class = result[0].probs.top1
        confidence = result[0].probs.top1conf.item()
        true_class = 0 if 'fall' in str(img_path.parent).lower() else 1
        
        print(f"Image: {img_path.name}")
        print(f"  True: {CLASS_NAMES[true_class]}")
        print(f"  Predicted: {CLASS_NAMES[pred_class]} (confidence: {confidence:.3f})")
        print(f"  Correct: {'✅' if pred_class == true_class else '❌'}")
        print()
    except Exception as e:
        print(f"Error processing {img_path.name}: {e}")
        continue

New https://pypi.org/project/ultralytics/8.3.167 available  Update with 'pip install -U ultralytics'


🧹 Removing cache file: dataset\train.cache
🧹 Removing cache file: dataset\val.cache
Using Torch version: 2.0.1+cpu
CUDA available: False


Ultralytics YOLOv8.0.197  Python-3.10.8 torch-2.0.1+cpu CPU (Intel Core(TM) i3-10100F 3.60GHz)
engine\trainer: task=classify, mode=train, model=yolov8n-cls.pt, data=dataset, epochs=25, patience=10, batch=16, imgsz=224, save=True, save_period=-1, cache=False, device=cpu, workers=8, project=ActivityRecogYOLOv8, name=classification_v1, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, show=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, vid_stride=1, stream_buffer=False, line_width=None, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, boxes=True, format=torchscrip

📊 Validation Results: ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002A1DCE4ADD0>
curves: []
curves_results: []
fitness: 0.9523809552192688
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.9047619104385376, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9523809552192688}
save_dir: WindowsPath('ActivityRecogYOLOv8/classification_v12')
speed: {'preprocess': 0.0, 'inference': 8.57147148677281, 'loss': 0.0, 'postprocess': 0.0}
task: 'classify'
top1: 0.9047619104385376
top5: 1.0



image 1/1 c:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\test\fall\F01_11.jpg: 224x224 fall 1.00, normal 0.00, 17.0ms
Speed: 1.0ms preprocess, 17.0ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)


🧠 Predicted class for F01_11.jpg: 0


In [11]:
# 📌 1. Setup
import os
import cv2
import torch
import shutil
from pathlib import Path
from ultralytics import YOLO

# ⚙️ Parameters
DATASET_PATH = Path('dataset')
PROJECT_NAME = 'ActivityRecogYOLOv8'
EXPERIMENT_NAME = 'classification_v1'
EPOCHS = 25
IMG_SIZE = 224
BATCH = 16

# 🧹 Clean old .cache files if exist
for split in ['train', 'val', 'test']:
    split_path = DATASET_PATH / split
    cache_file = split_path.with_suffix('.cache')
    if cache_file.exists():
        print(f"🧹 Removing cache file: {cache_file}")
        cache_file.unlink()

# 📦 Check PyTorch + CUDA
print(f"Using Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# ✅ 2. Load YOLOv8 Classification Model
cls_model = YOLO('yolov8n-cls.pt')  # nano model for faster training

# ✅ 3. Train
results = cls_model.train(
    data=str(DATASET_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=PROJECT_NAME,
    name=EXPERIMENT_NAME,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    patience=10,
    verbose=True,
    cache=False
)

# ✅ 4. Validate
metrics = cls_model.val()
print("📊 Validation Results:", metrics)

# ✅ 5. Predict (Single Image)
sample_test_img = list((DATASET_PATH / "test").rglob("*.jpg"))[0]
pred = cls_model.predict(source=str(sample_test_img), imgsz=IMG_SIZE)
print(f"🧠 Predicted class for {sample_test_img.name}: {pred[0].probs.top1}")

# ✅ 6. Real-time Webcam Inference (with Human Detection Box)

# Load detection model for humans (YOLOv8n)
det_model = YOLO('yolov8n.pt')

# Load trained classification model
cls_model = YOLO(f'{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt')

print("🚀 Starting real-time webcam detection...")
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Could not open webcam.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = det_model(frame, classes=[0], conf=0.5)  # class 0 = person
    person_detected = False

    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            if cls_id == 0:  # person
                person_detected = True
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # Crop and classify
                person_crop = frame[y1:y2, x1:x2]
                if person_crop.size == 0:
                    continue

                resized = cv2.resize(person_crop, (IMG_SIZE, IMG_SIZE))
                result = cls_model.predict(source=resized, imgsz=IMG_SIZE, verbose=False)
                activity = result[0].names[result[0].probs.top1]

                # Draw label
                cv2.putText(frame, f'{activity} ({conf:.2f})', (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    if not person_detected:
        cv2.putText(frame, 'No Person Detected', (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow('Real-Time Activity Recognition', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


New https://pypi.org/project/ultralytics/8.3.167 available  Update with 'pip install -U ultralytics'


Using Torch version: 2.0.1+cpu
CUDA available: False


Ultralytics YOLOv8.0.197  Python-3.10.8 torch-2.0.1+cpu CPU (Intel Core(TM) i3-10100F 3.60GHz)
engine\trainer: task=classify, mode=train, model=yolov8n-cls.pt, data=dataset, epochs=25, patience=10, batch=16, imgsz=224, save=True, save_period=-1, cache=False, device=cpu, workers=8, project=ActivityRecogYOLOv8, name=classification_v1, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, show=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, vid_stride=1, stream_buffer=False, line_width=None, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, boxes=True, format=torchscrip

📊 Validation Results: ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002A1D1A3A980>
curves: []
curves_results: []
fitness: 0.9078947305679321
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8157894611358643, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9078947305679321}
save_dir: WindowsPath('ActivityRecogYOLOv8/classification_v14')
speed: {'preprocess': 0.0, 'inference': 5.894209209241365, 'loss': 0.0, 'postprocess': 0.0}
task: 'classify'
top1: 0.8157894611358643
top5: 1.0



image 1/1 c:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\test\fall\images\F01_11.jpg: 224x224 fall 1.00, normal 0.00, 17.0ms
Speed: 3.0ms preprocess, 17.0ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)


🧠 Predicted class for F01_11.jpg: 0
🚀 Starting real-time webcam detection...



0: 480x640 (no detections), 165.0ms
Speed: 2.0ms preprocess, 165.0ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 155.1ms
Speed: 3.0ms preprocess, 155.1ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 146.1ms
Speed: 3.0ms preprocess, 146.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 158.5ms
Speed: 3.0ms preprocess, 158.5ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 151.3ms
Speed: 3.0ms preprocess, 151.3ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 143.0ms
Speed: 3.0ms preprocess, 143.0ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 140.1ms
Speed: 2.0ms preprocess, 140.1ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 175.1ms
Speed: 3.0ms preprocess, 175.1ms infere

In [13]:
# 📌 Setup
import cv2
import torch
from ultralytics import YOLO
import numpy as np

# ⚙️ Load models
det_model = YOLO("yolov8n.pt")         # YOLOv8 Nano - for person detection
cls_model = YOLO("ActivityRecogYOLOv8/classification_v1/weights/best.pt")  # Your trained classification model

# ✅ Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 🧠 Activity labels (update based on your training classes)
activity_labels = cls_model.model.names

# 📷 Start Webcam
cap = cv2.VideoCapture(0)  # Use 0 for built-in cam or replace with file path

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_resized = cv2.resize(frame, (640, 480))
    results = det_model.predict(source=frame_resized, classes=[0], conf=0.5, device=device, verbose=False)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy().astype(int)  # x1, y1, x2, y2
        for box in boxes:
            x1, y1, x2, y2 = box
            person_crop = frame_resized[y1:y2, x1:x2]

            # Only classify if crop is valid
            if person_crop.size > 0:
                crop_resized = cv2.resize(person_crop, (224, 224))
                crop_rgb = cv2.cvtColor(crop_resized, cv2.COLOR_BGR2RGB)

                # Classification
                result_cls = cls_model.predict(source=crop_rgb, imgsz=224, device=device, verbose=False)[0]
                label_id = int(result_cls.probs.top1)
                label = activity_labels[label_id]
                confidence = result_cls.probs.data[label_id].item()

                # Draw box & label
                color = (0, 255, 0) if label != "fall" else (0, 0, 255)
                cv2.rectangle(frame_resized, (x1, y1), (x2, y2), color, 2)
                cv2.putText(
                    frame_resized,
                    f'{label} ({confidence:.2f})',
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    color,
                    2
                )

    # 🖼️ Show output
    cv2.imshow("Real-Time Activity Detection", frame_resized)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Using device: cpu


In [14]:
# ✅ WEDAFALL - Full Pipeline: Classification + Real-Time Inference & Logging

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO
import cv2
import logging
import requests
import torch
import yaml
from datetime import datetime

# ✅ Step 1: Setup Logging and Versions
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.info(f"Python version: {sys.version}")
logging.info(f"Torch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

# ✅ Step 2: Config
DATASET_PATH = Path("dataset")
CLASS_NAMES = ['fall', 'normal']
PROJECT_NAME = "wedafall_yolo_v2"
EXPERIMENT_NAME = "fall_classifier"
IMG_SIZE = 224
EPOCHS = 30
BATCH = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FALL_LOG_DIR = Path("fall_logs")
FALL_LOG_DIR.mkdir(exist_ok=True)

# ✅ Step 3: Download Pretrained Model if Missing
def download_yolo_model(model_name="yolov8n-cls.pt"):
    model_path = Path(model_name)
    if not model_path.exists():
        url = f"https://github.com/ultralytics/assets/releases/download/v0.0.0/{model_name}"
        logging.info(f"Downloading model from {url}")
        response = requests.get(url)
        with open(model_path, 'wb') as f:
            f.write(response.content)
    return model_path

# ✅ Step 4: Create and Validate Dataset
def create_data_yaml(dataset_path):
    data_yaml = dataset_path / "data.yaml"
    content = {
        "path": str(dataset_path.resolve()),
        "train": "train",
        "val": "val",
        "test": "test",
        "nc": len(CLASS_NAMES),
        "names": CLASS_NAMES
    }
    with open(data_yaml, 'w') as f:
        yaml.dump(content, f)
    return data_yaml

def verify_dataset(dataset_path):
    required_dirs = [dataset_path / split / cls for split in ["train", "val", "test"] for cls in CLASS_NAMES]
    for d in required_dirs:
        if not d.exists():
            raise FileNotFoundError(f"Missing: {d}")
    data_yaml = dataset_path / "data.yaml"
    if not data_yaml.exists():
        create_data_yaml(dataset_path)
    return data_yaml

# ✅ Step 5: Train Model
def train_model(data_yaml):
    base_model = download_yolo_model()
    model = YOLO(str(base_model))
    model.train(
        data=str(DATASET_PATH),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        project=PROJECT_NAME,
        name=EXPERIMENT_NAME,
        device=DEVICE,
        patience=10,
        verbose=True,
        cache=False
    )
    return f"{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt"

# ✅ Step 6: Evaluate Model
def evaluate_model(model_path):
    model = YOLO(model_path)
    all_images = list((DATASET_PATH / "test").rglob("*.jpg")) + list((DATASET_PATH / "test").rglob("*.png"))
    y_true, y_pred = [], []

    for img_path in all_images:
        result = model.predict(source=str(img_path), device=DEVICE, verbose=False)[0]
        pred_class = int(result.probs.top1)
        y_pred.append(pred_class)
        true_class = 0 if 'fall' in str(img_path).lower() else 1
        y_true.append(true_class)

    print("\n📊 Classification Report:\n", classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    print(f"✅ Accuracy: {np.mean(np.array(y_true) == np.array(y_pred)) * 100:.2f}%")

# ✅ Step 7: Real-Time Fall Detection with Object Box
def run_realtime_inference(classifier_model_path):
    detector = YOLO("yolov8n.pt")  # Object detector
    classifier = YOLO(classifier_model_path)

    cap = cv2.VideoCapture(0)
    logging.info("🟢 Starting webcam...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = detector(frame, classes=[0], conf=0.5, device=DEVICE)  # detect humans only
        for r in results:
            boxes = r.boxes.xyxy.cpu().numpy().astype(int)
            for box in boxes:
                x1, y1, x2, y2 = box
                person_crop = frame[y1:y2, x1:x2]
                if person_crop.size == 0:
                    continue
                resized = cv2.resize(person_crop, (224, 224))
                rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)

                pred = classifier.predict(rgb, imgsz=224, device=DEVICE, verbose=False)[0]
                cls_id = int(pred.probs.top1)
                conf = float(pred.probs.top1conf)
                label = CLASS_NAMES[cls_id]
                color = (0, 0, 255) if label == "fall" else (0, 255, 0)

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"{label} ({conf:.2f})", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                if label == "fall" and conf > 0.85:
                    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                    filepath = FALL_LOG_DIR / f"fall_{timestamp}.jpg"
                    cv2.imwrite(str(filepath), frame)
                    logging.warning(f"🚨 Fall detected! Saved frame at {filepath}")

        cv2.imshow("WEDAFALL - Real-time Detection", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

# ✅ Step 8: Main Execution
if __name__ == "__main__":
    data_yaml = verify_dataset(DATASET_PATH)
    best_model = train_model(data_yaml)
    evaluate_model(best_model)
    run_realtime_inference(best_model)


2025-07-18 02:05:46,602 - INFO - Python version: 3.10.8 (tags/v3.10.8:aaaf517, Oct 11 2022, 16:50:30) [MSC v.1933 64 bit (AMD64)]
2025-07-18 02:05:46,603 - INFO - Torch version: 2.0.1+cpu, CUDA available: False
New https://pypi.org/project/ultralytics/8.3.167 available  Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.0.197  Python-3.10.8 torch-2.0.1+cpu CPU (Intel Core(TM) i3-10100F 3.60GHz)
engine\trainer: task=classify, mode=train, model=yolov8n-cls.pt, data=dataset, epochs=30, patience=10, batch=16, imgsz=224, save=True, save_period=-1, cache=False, device=cpu, workers=8, project=wedafall_yolo_v2, name=fall_classifier, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half


📊 Classification Report:
               precision    recall  f1-score   support

        fall       1.00      1.00      1.00        21
      normal       1.00      1.00      1.00        21

    accuracy                           1.00        42
   macro avg       1.00      1.00      1.00        42
weighted avg       1.00      1.00      1.00        42



<Figure size 640x480 with 2 Axes>

✅ Accuracy: 100.00%


2025-07-18 02:10:25,572 - INFO - 🟢 Starting webcam...

0: 480x640 (no detections), 124.5ms
Speed: 1.0ms preprocess, 124.5ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 140.5ms
Speed: 5.0ms preprocess, 140.5ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 132.4ms
Speed: 3.5ms preprocess, 132.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 123.5ms
Speed: 2.0ms preprocess, 123.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 140.0ms
Speed: 3.0ms preprocess, 140.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 125.5ms
Speed: 3.0ms preprocess, 125.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 132.5ms
Speed: 2.5ms preprocess, 132.5ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)



In [2]:
import os
import cv2
import torch
from pathlib import Path
from ultralytics import YOLO, __version__

# 📁 Parameters
DATASET_PATH = Path('dataset')
PROJECT_NAME = 'ActivityRecogYOLOv8_v2'
EXPERIMENT_NAME = 'classification_v1'
EPOCHS = 25
IMG_SIZE = 224
BATCH = 16

# ✅ Environment check
print(f"🧠 PyTorch Version: {torch.__version__}")
print(f"🧠 Ultralytics Version: {__version__}")
print(f"🖥️ CUDA Available: {torch.cuda.is_available()}")
print(f"📁 Dataset Path Exists: {DATASET_PATH.exists()}")

# ⚙️ Clear previous corrupted .cache files
for split in ['train', 'val', 'test']:
    split_path = DATASET_PATH / split
    cache_file = split_path.with_suffix('.cache')
    if cache_file.exists():
        print(f"🧹 Removing cache file: {cache_file}")
        cache_file.unlink()

# ✅ Load YOLOv8 Classification Model
try:
    model_cls = YOLO('yolov8n-cls.pt')  # nano model for speed
    print("✅ Model loaded successfully")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    exit()

# ✅ Train Model
try:
    results = model_cls.train(
        data=str(DATASET_PATH),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        project=PROJECT_NAME,
        name=EXPERIMENT_NAME,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        patience=10,
        verbose=True,
        cache=False
    )
except Exception as e:
    print(f"❌ Error during training: {e}")
    exit()

# ✅ Validate Model
metrics = model_cls.val()
print("📊 Validation Results:", metrics)

# ✅ Save Final Model
final_model_path = f"{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt"
print(f"✅ Model saved to: {final_model_path}")

# ✅ Load Final Model for Inference
model_cls = YOLO(final_model_path)

# ✅ Test Prediction (Single image)
try:
    sample_img = list((DATASET_PATH / "test").rglob("*.jpg"))[0]
    pred = model_cls.predict(source=str(sample_img), imgsz=IMG_SIZE)
    print(f"📌 Sample Prediction for {sample_img.name}: {pred[0].probs.top1conf:.2f} - {pred[0].names[pred[0].probs.top1]}")
except Exception as e:
    print(f"❌ Error during prediction: {e}")

# ✅ Real-time Inference with Webcam + Object Box
print("📷 Starting Real-Time Inference... Press 'q' to exit")
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Preprocess for classification (resize + center)
    input_frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))

    # Save temp image
    cv2.imwrite("temp.jpg", input_frame)

    # Predict on current frame
    try:
        result = model_cls.predict(source="temp.jpg", imgsz=IMG_SIZE, save=False)[0]
        pred_label = result.names[result.probs.top1]
        conf_score = result.probs.top1conf

        # Add visual overlays
        label_text = f"{pred_label} ({conf_score*100:.1f}%)"
        color = (0, 255, 0) if "normal" in pred_label.lower() else (0, 0, 255)
        cv2.rectangle(frame, (20, 20), (320, 70), color, -1)
        cv2.putText(frame, label_text, (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

        cv2.imshow("🧠 Real-Time Fall Detection", frame)
    except Exception as e:
        print(f"❌ Error during real-time inference: {e}")

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
os.remove("temp.jpg")  # Clean up temp file

🧠 PyTorch Version: 2.7.1+cpu
🧠 Ultralytics Version: 8.3.167
🖥️ CUDA Available: False
📁 Dataset Path Exists: True
🧹 Removing cache file: dataset\train.cache
🧹 Removing cache file: dataset\val.cache
✅ Model loaded successfully
Ultralytics 8.3.167  Python-3.10.8 torch-2.7.1+cpu CPU (Intel Core(TM) i3-10100F 3.60GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n

train: Scanning C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\train... 184 images, 0 corrupt: 100%|██████████| 184/184 [00:00<00:00, 3016.34it/s]

train: New cache created: C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\train.cache
val: Fast image access  (ping: 0.10.0 ms, read: 1158.3242.9 MB/s, size: 438.8 KB)



c:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\.venv\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\val... 38 images, 0 corrupt: 100%|██████████| 38/38 [00:00<00:00, 3454.87it/s]

val: New cache created: C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001667, momentum=0.9) with parameter groups 26 weight(decay=0.0), 27 weight(decay=0.0005), 27 bias(decay=0.0)
Image sizes 224 train, 224 val
Using 0 dataloader workers
Logging results to ActivityRecogYOLOv8_v2\classification_v1
Starting training for 25 epochs...

      Epoch    GPU_mem       loss  Instances       Size


...

Optimizer stripped from ActivityRecogYOLOv8_v2\classification_v1\weights\last.pt, 3.0MB
Optimizer stripped from ActivityRecogYOLOv8_v2\classification_v1\weights\best.pt, 3.0MB

Validating ActivityRecogYOLOv8_v2\classification_v1\weights\best.pt...
Ultralytics 8.3.167  Python-3.10.8 torch-2.7.1+cpu CPU (Intel Core(TM) i3-10100F 3.60GHz)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\train... found 184 images in 2 classes  
val: C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\val... found 38 images in 2 classes  
test: C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\test... found 42 images in 2 classes  


               classes   top1_acc   top5_acc: 100%|██████████| 2/2 [00:01<00:00,  1.39it/s]


                   all          1          1
Speed: 0.0ms preprocess, 5.3ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to ActivityRecogYOLOv8_v2\classification_v1
Ultralytics 8.3.167  Python-3.10.8 torch-2.7.1+cpu CPU (Intel Core(TM) i3-10100F 3.60GHz)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\train... found 184 images in 2 classes  
val: C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\val... found 38 images in 2 classes  
test: C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\test... found 42 images in 2 classes  
val: Fast image access  (ping: 0.10.0 ms, read: 1563.0185.6 MB/s, size: 438.8 KB)


val: Scanning C:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\val... 38 images, 0 corrupt: 100%|██████████| 38/38 [00:00<?, ?it/s]
c:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\.venv\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
               classes   top1_acc   top5_acc: 100%|██████████| 3/3 [00:01<00:00,  2.25it/s]


                   all          1          1
Speed: 0.0ms preprocess, 5.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to ActivityRecogYOLOv8_v2\classification_v12
📊 Validation Results: ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000024307E46BC0>
curves: []
curves_results: []
fitness: 1.0
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 1.0, 'metrics/accuracy_top5': 1.0, 'fitness': 1.0}
save_dir: WindowsPath('ActivityRecogYOLOv8_v2/classification_v12')
speed: {'preprocess': 0.0014263157362093854, 'inference': 5.1418631580753225, 'loss': 0.00014473679536757502, 'postprocess': 0.00043157897858978495}
task: 'classify'
top1: 1.0
top5: 1.0
✅ Model saved to: ActivityRecogYOLOv8_v2/classification_v1/weights/best.pt

image 1/1 c:\Users\Madye\Desktop\TimeSeriesAnomalyDetection\Yolo\dataset\test\fall\images\F01_11.jpg: 224x224 fall